# White advantage on Lichess: run in Colab

Streams the Lichess monthly files straight from database.lichess.org into `white_advantage`,
so nothing big is saved to disk and memory use stays tiny.

Run the cells in order. At the end you download `results.zip`. Put `output.csv` and
`summary.txt` in `output/`, and `report_numbers.tex` and `checkpointOne.pdf` in
`reports/checkpoint1/`, then push.

Keep this tab open while it runs. Colab can stop an idle or very long session.

## 1. Install zstd and compile the program

In [ ]:
!apt-get -qq update > /dev/null
!apt-get -qq install -y libzstd-dev > /dev/null
!rm -rf repo && git clone -q https://github.com/Sahil-M-Deo/The-Perks-of-Being-a-First-Mover.git repo
!g++ -O2 -std=c++17 -pthread repo/scripts/white_advantage.cpp -o repo/scripts/white_advantage -lzstd && echo "compiled" 

## 2. Install LaTeX (so the program can also build the PDF)

Takes a few minutes. You can skip it: the program still writes `report_numbers.tex`,
and you can build the PDF later on your laptop or Overleaf.

In [ ]:
!apt-get -qq install -y texlive-latex-recommended texlive-latex-extra texlive-fonts-recommended texlive-pictures > /dev/null && echo "LaTeX installed" 

## 3. Choose the months

`MAX_GAMES = 0` reads every game. Try `MAX_GAMES = 1000000` first for a quick test run.

In [ ]:
MONTHS = ["2024-01", "2024-02"]
MAX_GAMES = 0

## 4. Run

Each month is downloaded and read at the same time through a pipe (`mkfifo`),
so the compressed file never sits on disk. This takes a while: it is limited by download speed.

**Check after it finishes:** the "games read" for each month should match the Games column on
database.lichess.org. If it is lower, the download was cut off partway; rerun the cell.

In [ ]:
files = [f"lichess_db_standard_rated_{m}.pgn.zst" for m in MONTHS]
lines = ["cd repo/output", "rm -f lichess_db_*.pgn.zst"]
for f in files:
    # a named pipe with the real file name, so the program knows the month and that it is .zst
    lines.append(f"mkfifo {f}")
    # -f: on an HTTP error write nothing, instead of feeding an HTML error page to zstd
    lines.append(f"curl -fsSL https://database.lichess.org/standard/{f} > {f} &")
extra = f"--max-games {MAX_GAMES}" if MAX_GAMES > 0 else ""
lines.append(f"../scripts/white_advantage {' '.join(files)} {extra} --report ../reports/checkpoint1/report_numbers.tex | tee summary.txt")
lines.append("rm -f lichess_db_*.pgn.zst")
open("run.sh", "w").write("\n".join(lines) + "\n")

# run with ! (not subprocess) so the output shows up live in the notebook
!bash run.sh

## 5. Download the results

In [ ]:
!cd repo && zip -q -j ../results.zip output/output.csv output/summary.txt reports/checkpoint1/report_numbers.tex reports/checkpoint1/checkpointOne.pdf 2>/dev/null; ls -la ../results.zip
from google.colab import files as colab_files
colab_files.download("results.zip")